# tt-mlir #8570 repro check

런타임: CPU 고RAM 선택 → **런타임 → 모두 실행**만 하면 됩니다. 완료되면 GitHub에 저장(File → Save a copy in GitHub)해주세요.

In [1]:
%cd /content
!rm -rf /content/tt-mlir
!git clone --branch main https://github.com/alexxony/tt-mlir.git /content/tt-mlir
%cd /content/tt-mlir
!git remote add upstream https://github.com/tenstorrent/tt-mlir.git
!git fetch upstream main --quiet
!git reset --hard upstream/main
!git log --oneline -1

/content
Cloning into '/content/tt-mlir'...
remote: Enumerating objects: 279824, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 279824 (delta 13), reused 9 (delta 5), pack-reused 279795 (from 1)
Receiving objects: 100% (279824/279824), 92.84 MiB | 27.37 MiB/s, done.
Resolving deltas: 100% (218025/218025), done.
/content/tt-mlir
HEAD is now at 70b7117e56 [tt-crank] decompose slice_scatter onto slice/cat instead of gather + where (#9325)
70b7117e56 (HEAD -> main, upstream/main) [tt-crank] decompose slice_scatter onto slice/cat instead of gather + where (#9325)


In [ ]:
%%bash
set -e
sudo mkdir -p /opt/ttmlir-toolchain
sudo chown -R $(whoami) /opt/ttmlir-toolchain
apt-get -qq update && apt-get -qq install -y ninja-build clang lld ccache python3.12-venv > /tmp/apt.log 2>&1
cd /content/tt-mlir
source env/activate
cmake -B env/build env -G Ninja -DCMAKE_C_COMPILER=clang -DCMAKE_CXX_COMPILER=clang++ 2>&1 | tail -40
cmake --build env/build 2>&1 | tail -100


In [3]:
%%bash
set -e
cd /content/tt-mlir
source env/activate
cmake -B build . \
    -G Ninja \
    -DCMAKE_BUILD_TYPE=Release \
    -DCMAKE_C_COMPILER=clang -DCMAKE_CXX_COMPILER=clang++ \
    -DTTMLIR_ENABLE_RUNTIME=OFF \
    -DTTMLIR_ENABLE_RUNTIME_TESTS=OFF \
    -DTTMLIR_ENABLE_PYKERNEL=OFF \
    -DTT_RUNTIME_ENABLE_PERF_TRACE=OFF \
    -DTTMLIR_ENABLE_OPMODEL=OFF \
    -DTTMLIR_ENABLE_BINDINGS_PYTHON=OFF 2>&1 | tail -60


  tt-mlir might not be fully/correctly configured, have you run environment setup?
  For more information refer to: https://docs.tenstorrent.com/tt-mlir/getting-started.html
-- The CXX compiler identification is Clang 18.1.3
-- The C compiler identification is Clang 18.1.3
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/clang++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/clang - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Using LLD linker: /usr/bin/ld.lld-18
CMake Error at cmake/modules/FindMLIR.cmake:2 (find_package):
  Could not find a package configuration file provided by "MLIR" with any of
  the following names:

    MLIRConfig.cmake
    mlir-config.cmake

  Add the installation prefix of "MLIR" to CMAKE_PREFIX_PATH or

In [4]:
%%bash
cd /content/tt-mlir
source env/activate
ninja -C build ttmlir-opt 2>&1 | tail -100


  tt-mlir might not be fully/correctly configured, have you run environment setup?
  For more information refer to: https://docs.tenstorrent.com/tt-mlir/getting-started.html
ninja: Entering directory `build'
ninja: error: loading 'build.ninja': No such file or directory


In [5]:
import subprocess
r = subprocess.run(["/content/tt-mlir/build/bin/ttmlir-opt", "--version"], capture_output=True, text=True, timeout=30)
print("rc:", r.returncode)
print(r.stdout)
print(r.stderr)

FileNotFoundError: [Errno 2] No such file or directory: '/content/tt-mlir/build/bin/ttmlir-opt'

In [ ]:
repro_mlir = r"""// SPDX-FileCopyrightText: (c) 2026 Tenstorrent AI ULC
//
// SPDX-License-Identifier: Apache-2.0

// REQUIRES: stablehlo
// RUN: ttmlir-opt --convert-stablehlo-to-ttir %s | FileCheck %s

// Repro for https://github.com/tenstorrent/tt-mlir/issues/8570
// scatter with leading update_window_dims=[0] (Qwen 3.5 27B MRoPE position_ids pattern)

module @SyncTensorsGraph.45 attributes {mhlo.is_dynamic = false} {
  func.func @main(%arg0: tensor<3x1x494xi64>,
                  %arg1: tensor<3x494xi64>,
                  %arg2: tensor<494xi64>) -> tensor<3x1x494xi64> {
    %c   = stablehlo.constant dense<494> : tensor<494xi64>
    %c_0 = stablehlo.constant dense<0>   : tensor<494xi64>
    %0 = stablehlo.reshape %arg0 : (tensor<3x1x494xi64>) -> tensor<3x494xi64>
    %1 = stablehlo.reshape %arg2 : (tensor<494xi64>)     -> tensor<1x1x494xi64>
    %2 = stablehlo.reshape %1    : (tensor<1x1x494xi64>) -> tensor<494xi64>
    %3 = stablehlo.compare LT, %2, %c_0 : (tensor<494xi64>, tensor<494xi64>) -> tensor<494xi1>
    %4 = stablehlo.add %2, %c    : tensor<494xi64>
    %5 = stablehlo.select %3, %4, %2 : tensor<494xi1>, tensor<494xi64>
    %6 = stablehlo.reshape %5    : (tensor<494xi64>)     -> tensor<494x1xi64>
    %7 = stablehlo.reshape %arg1 : (tensor<3x494xi64>)   -> tensor<1x3x494xi64>
    %8 = stablehlo.reshape %7    : (tensor<1x3x494xi64>) -> tensor<3x494xi64>
    %9 = "stablehlo.scatter"(%0, %6, %8) <{
           scatter_dimension_numbers = #stablehlo.scatter<
             update_window_dims = [0], inserted_window_dims = [1],
             scatter_dims_to_operand_dims = [1], index_vector_dim = 1>
         }> ({
      ^bb0(%a: tensor<i64>, %b: tensor<i64>):
        stablehlo.return %b : tensor<i64>
      }) : (tensor<3x494xi64>, tensor<494x1xi64>, tensor<3x494xi64>) -> tensor<3x494xi64>
    %10 = stablehlo.reshape %9 : (tensor<3x494xi64>) -> tensor<3x1x494xi64>
    return %10 : tensor<3x1x494xi64>
  }
}
"""
with open("/content/repro_8570.mlir", "w") as f:
    f.write(repro_mlir)
print("written")

In [ ]:
import subprocess
r = subprocess.run(
    ["/content/tt-mlir/build/bin/ttmlir-opt", "--convert-stablehlo-to-ttir", "/content/repro_8570.mlir"],
    capture_output=True, text=True, timeout=60,
)
print("=== rc:", r.returncode, "===")
print("=== stdout ===")
print(r.stdout)
print("=== stderr ===")
print(r.stderr)

## 결과 판독

- **rc != 0 + stderr에 `ttir.repeat` verifier 에러** → 버그 아직 살아있음, #8570 재현 확인, #8594 여전히 유효.
- **rc == 0** → 이미 고쳐져 있음, `ttir.scatter` 결과 IR이 stdout에 찍혀야 함. #8570은 사실상 해결된 상태.
- 이 셀 실행 후 **File → Save a copy in GitHub**로 저장 부탁드립니다 (브랜치: 이 노트북이 push된 브랜치 그대로).